In [22]:
import pandas as pd
import sqlite3
import random
import hashlib
import re


### Generate sample data

#### Create Stock CSV

In [23]:
raw_csv = """
barcode,espansione,nome,prezzo,quantita_stock,condizione,prezzo_acquisto
PIKA-SWS-GO-2B1AC4,SWSH039,Pikachu V Full Art,12.5,3,Good,10.23
CHAR-BUS-PL-2EAFE2,BUS-20,Charizard GX Shiny,45.0,1,Played,36.82
BLAS-XY1-NE-704B67,XY122,Blastoise EX Holo,18.75,5,Near Mint,15.34
VENU-SWS-PL-D47DEE,SWSH102,Venusaur VMAX Rainbow,22.3,2,Played,18.25
MEWT-SLG-EX-82C58C,SLG-78,Mewtwo GX Secret Rare,39.99,4,Excellent,32.72
GENG-FST-MI-C992C6,FST-253,Gengar V Alt Art,27.5,1,Mint,22.5
RAYQ-EVS-NE-8804B1,EVS-218,Rayquaza VMAX Hyper Rare,55.0,2,Near Mint,45.0
LUCA-SM1-NE-0B853F,SM158,Lucario EX Promo,9.9,6,Near Mint,8.1
SNOR-SWS-GO-9A44B1,SWSH142,Snorlax VMAX,14.2,3,Good,11.62
EEVE-SM2-GO-4F504D,SM233,Eevee GX Full Art,11.0,7,Good,9.0
DRAG-BRS-PL-472B6A,BRS-123,Dragonite VSTAR,19.8,2,Played,16.2
UMBR-EVS-PL-8F38C0,EVS-215,Umbreon VMAX Alt Art,120.0,1,Played,98.18
ARCE-BRS-GO-CC50EB,BRS-184,Arceus VSTAR Gold,65.0,2,Good,53.18
GREN-SV5-EX-FC50D6,SV-56,Greninja GX Shiny Vault,13.6,3,Excellent,11.13
TYRA-BST-GO-461FE3,BST-155,Tyranitar V Alt Art,28.4,2,Good,23.24

"""

with open("stock.csv", "w") as f:
    f.write(raw_csv)

In [24]:
def generate_barcode(nome, espansione, condizione):
    def clean(s):
        return re.sub(r"[^A-Z0-9]", "", s.upper())

    # parte leggibile
    base = f"{clean(nome)[:4]}-{clean(espansione)[:3]}-{clean(condizione)[:2]}"

    # hash deterministico
    raw = f"{nome}|{espansione}|{condizione}".upper()
    short_hash = hashlib.md5(raw.encode()).hexdigest()[:6].upper()

    return f"{base}-{short_hash}"


generate_barcode("Pikachu V Full Art", "SWSH039", "Excellent")


'PIKA-SWS-EX-4BF6D1'

In [27]:
df_stock = pd.read_csv("stock.csv")
df_stock["condizione"] = [
    random.choice(["Mint", "Near Mint", "Excellent", "Good", "Played"])
    for _ in range(len(df_stock))
]
df_stock["barcode"] = df_stock.apply(
    lambda row: generate_barcode(row["nome"], row["espansione"], row["condizione"]),
    axis=1,
)
df_stock["prezzo_acquisto"] = df_stock["prezzo"] * random.uniform(
    0.5, 0.9
)  # prezzo di acquisto inferiore al prezzo di vendita
df_stock["prezzo_acquisto"] = df_stock["prezzo_acquisto"].round(2)
df_stock["da_prezzare"] = ["No" for _ in range(len(df_stock))]
df_stock = df_stock[
    ["barcode", "espansione", "nome", "condizione", "prezzo", "quantita_stock",  "prezzo_acquisto", "da_prezzare"]
]
df_stock.to_csv("stock.csv", index=False)

#### Create stock db

In [28]:
conn = sqlite3.connect("pokemon.db")
cursor = conn.cursor()

# Leggi il CSV
df = pd.read_csv("stock.csv")

# Scrive la tabella nel database
df.to_sql(
    "stock",
    conn,
    if_exists="replace",  # "append" se vuoi aggiungere senza sovrascrivere
    index=False,
)
conn.close()

#### Create sales table

In [ ]:
conn = sqlite3.connect("pokemon.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS sales")

cursor.execute("""
CREATE TABLE sales (
    sale_id INTEGER PRIMARY KEY AUTOINCREMENT,
    barcode TEXT,
    espansione TEXT,
    nome TEXT,
    condizione TEXT,
    prezzo_stock REAL,
    prezzo_vendita REAL,
    sell_date TEXT
)
""")

conn.commit()
conn.close()

#### Create purchase table

In [21]:
conn = sqlite3.connect("pokemon.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS purchase")

cursor.execute("""
CREATE TABLE purchase (
    purchase_id INTEGER PRIMARY KEY AUTOINCREMENT,
    barcode TEXT,
    espansione TEXT,
    nome TEXT,
    condizione TEXT,
    prezzo_acquisto REAL,
    purchase_date TEXT
)
""")

conn.commit()
conn.close()

#### CREATE INVENTARY DB

In [18]:
conn = sqlite3.connect("card_database.db")
cursor = conn.cursor()

In [19]:
df = pd.read_csv(r"C:\Users\Salvo\Downloads\export-Pokemon-15-04-2026.csv")
df.rename(columns={"name": "nome", "expansionCode": "espansione"}, inplace=True)
df.drop(columns=[x for x in df.columns if x not in ["nome", "espansione"]], inplace=True)

In [20]:
df.to_sql(
    "stock",
    conn,
    if_exists="replace",  # "append" se vuoi aggiungere senza sovrascrivere
    index=False,
)
conn.close()